# Field validation — `kinematic` (SURF pipeline)

End-to-end validation of every calculated field in the **kinematic** surface subset: the notebook RUNs the pipeline, LOADs its own output, and validates each field with dependency-chain maps, PDFs, and a literature comparison.

| | |
|---|---|
| Subset | `kinematic` (SURF) |
| Timestep | 2012-11-09 12:00:00 (`20121109_120000`) |
| run_id | `field_validation_v1` |
| Plan | `prompts/field_validation.md` |
| Field reference | `docs/Fields.md` |

Velocity-gradient diagnostics.  Every computed step is shown: the four Jacobian components (du_dx, du_dy, dv_dx, dv_dy — tracer-point, tensor-rotated) are computed live and plotted as chain columns feeding each final field.

## Section 1 — RUN the SURF pipeline for this subset + timestep

One cell (pattern from
`notebooks/notebooks_global/running_generate_global_script.ipynb`).
Existing stores for this date are skipped unless `--clobber` is added,
so re-running is a cheap no-op.

In [ ]:
# Section 1: run the SURF pipeline for this subset and timestep.
SUBSET   = "kinematic"
PIPELINE = "SURF"
RUN_ID   = "field_validation_v1"
DATE     = "2012-11-09 12:00:00"   # single validation timestep

!generate-global \
    --config ../../../configs/global/run/field_validation_surface.yaml \
    --pipeline $PIPELINE \
    --subset $SUBSET \
    --run_id $RUN_ID

## Section 2 — LOAD the data generated in Section 1

One cell (pattern from
`notebooks/notebooks_global/assess_generate_global_script.ipynb`):
product reader + grid reader, with shape/date confirmation printout.

In [ ]:
# Section 2: load the store written in Section 1 + the shared grid.
import numpy as np
import matplotlib.pyplot as plt

import dbof.io.filesystems as filesystems
import dbof.global_dataset_creation.zarr_dataset_global as zarr_dataset
import dbof.global_dataset_creation.zarr_grid_global as zarr_grid
from dbof.global_dataset_creation.subset_definitions import (
    get_subset_definition,
)

S3_ENDPOINT = "https://s3-west.nrp-nautilus.io"
BUCKET      = "dbof"
FOLDER      = "surface_fields"       # SURF output folder
DATE_PREFIX = "20121109_120000"      # matches DATE in Section 1

defn = get_subset_definition(PIPELINE, SUBSET)
# get_subset_definition already folds per-pipeline extras (e.g.
# oceQnet for SURF) into model_data_feature_channels.
CHANNELS = (list(defn["model_data_feature_channels"])
            + list(defn["compute_features_channels"]))

fs, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
reader = zarr_dataset.GlobalZarrDatasetReader(
    bucket=BUCKET, folder=FOLDER, run_id=RUN_ID,
    dataset_name=defn["dataset_name"], date_prefix=DATE_PREFIX, fs=fs,
)

fs_grid, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
grid_reader = zarr_grid.GlobalGridZarrReader(
    bucket=BUCKET, folder="LLC4320_GRID_2D",
    dataset_name="llc4320_grid.zarr", fs=fs_grid,
)
XC, YC = grid_reader.lon, grid_reader.lat

print(reader)
print(f"channels  : {reader.channel_names}")
print(f"shape     : {reader.shape}  (C, H, W)")
print(f"iteration : {reader.iteration}")
print(f"grid      : XC {XC.shape}, "
      f"lon [{XC.min():.1f}, {XC.max():.1f}], "
      f"lat [{YC.min():.1f}, {YC.max():.1f}]")

## Section 3 — SUBSET: `kinematic`

Channels (verbatim from `subset_definitions.SURFACE_SUBSETS`):

computed — `relative_vorticity`, `strain_n`, `strain_s`, `strain_mag`, `divergence`, `coriolis_f`, `rossby_number`, `okubo_weiss`

In [ ]:
# Section 3: guard — store channels must match the code definition.
print(f"subset '{SUBSET}': {len(CHANNELS)} channels")
for ch in CHANNELS:
    print(f"  - {ch}")
assert set(reader.channel_names) == set(CHANNELS), (
    "store channels differ from subset_definitions — regenerate the "
    "store (Section 1, --clobber) or check the code definition")
print("OK: store channel set matches subset_definitions")

## Section 4 — Field & dependency table

| FIELD NAME | UNITS | EQUATION | DEPENDENCIES | LOCATION OF CALC IN CODE |
|---|---|---|---|---|
| relative_vorticity | s⁻¹ | ζ = ∂v/∂x − ∂u/∂y | dv_dx, du_dy (← J ← raw U, V) | `calculate_fields.relative_vorticity` → `native_gradient.calculate_jacobian` |
| strain_n | s⁻¹ | σₙ = ∂u/∂x − ∂v/∂y | du_dx, dv_dy | `calculate_fields.strain` |
| strain_s | s⁻¹ | σₛ = ∂v/∂x + ∂u/∂y | dv_dx, du_dy | `calculate_fields.strain` |
| strain_mag | s⁻¹ | σ = √(σₙ² + σₛ²) | strain_n, strain_s | `calculate_fields.strain` |
| divergence | s⁻¹ | δ = ∂u/∂x + ∂v/∂y | du_dx, dv_dy | `calculate_fields.divergence` |
| coriolis_f | s⁻¹ | f = 2Ω sin(lat) | YC | `calculate_fields.coriolis_parameter` |
| rossby_number | — | Ro = ζ/f | relative_vorticity, coriolis_f | `calculate_fields.rossby_number` |
| okubo_weiss | s⁻² | W = σₙ² + σₛ² − ζ² | strain_mag, relative_vorticity | `calculate_fields.okubo_weiss_parameter` |

Live-plotted intermediates: rotated tracer-point `U`, `V`
(`calculate_fields.geographic_velocity`) and the Jacobian components
`du_dx`, `du_dy`, `dv_dx`, `dv_dy`
(`calculate_fields.compute_velocity_jacobian` →
`native_gradient.calculate_jacobian`).

Processing operations: land masking; staggered→tracer interpolation;
CS/SN tensor rotation of J; native-grid differentiation (halo rim);
f-division (Ro — equatorial extremes expected); face→lat-lon
stitching; global downsampling.

### Raw inputs & Jacobian components computed live

The store holds only derived channels; the chains start from the rotated tracer-point velocities and the four Jacobian components, computed here from the same OSN snapshot with the same code and batch-stitched.

In [ ]:
# Live raw inputs + Jacobian components (pipeline loaders/code).
import dbof.preprocessing.calculate_fields as calculate_fields
from dbof.cli.generate_global import load_snapshot
from dbof.global_dataset_creation.data_sources import get_data_source
from dbof.global_dataset_creation.grid_setup import set_up_grid
from dbof.utils.faces_to_latlon import stitch_and_mask

ds_grid, land_mask, xgrid = set_up_grid(PIPELINE, None)
ds_raw, ds_merge, it = load_snapshot(
    PIPELINE, DATE, ds_grid, ["U", "V", "Theta", "Salt"],
    surface_only=False, data_source=get_data_source(PIPELINE),
)
print(f"OSN iteration {it}  (store iteration {reader.iteration})")
for _v in ds_merge.data_vars:
    if ds_merge[_v].ndim >= 2:
        print(f"  raw {_v}: {ds_merge[_v].dtype}")

u_east, v_north = calculate_fields.geographic_velocity(
    ds_merge, xgrid)
J = calculate_fields.compute_velocity_jacobian(ds_merge, xgrid)

# |grad b| for the Balwada gradb-conditioned JPDF (Section 6).
bg = calculate_fields.compute_buoyancy_gradients(ds_merge, xgrid)

# Extra slice region for the Bachman/Balwada comparisons.
EXTRA_REGIONS = ["kerguelen"]

live_map = {
    "U": u_east, "V": v_north,
    "du_dx": J.du_dx, "du_dy": J.du_dy,
    "dv_dx": J.dv_dx, "dv_dy": J.dv_dy,
    "gradb_mag": (bg.zonal**2 + bg.merid**2) ** 0.5,
}
# Batch-stitch the live fields and slice to the validation domains
# immediately — at most BATCH full-res arrays alive at once.
from dbof.plotting import regions

SLICE_REGIONS = (regions.REGION_ORDER
                 + globals().get("EXTRA_REGIONS", []))
BATCH = 4
mask = {"_land_mask": (ds_merge.hFacC == 0)}
names = list(live_map)
region_arrays = {}
for i0 in range(0, len(names), BATCH):
    grp = names[i0:i0 + BATCH]
    ds_conv = ds_raw.assign({n: live_map[n] for n in grp})[grp]
    chw = stitch_and_mask(ds_conv, grp, mask)
    for k, n in enumerate(grp):
        region_arrays[n] = regions.select_all_regions(
            chw[k], XC, YC, names=SLICE_REGIONS)
    del chw
    print(f"stitched + sliced: {grp}")
print(f"live fields ready: {names}")

In [ ]:
# Slice the STORE channels to the validation domains (live fields, if
# any, were sliced in the previous cell).  Full-res arrays released
# immediately after slicing.
from dbof.plotting import regions
from dbof.plotting.field_cmaps import load_field_cmaps

CMAP_CFG, DIVERGING = load_field_cmaps()

# region_arrays[field][region] = (x, y, arr)
region_arrays = globals().get("region_arrays", {})
SLICE_REGIONS = (regions.REGION_ORDER
                 + globals().get("EXTRA_REGIONS", []))
for ch in CHANNELS:
    arr = reader.get_channel_snapshot(ch)
    region_arrays[ch] = regions.select_all_regions(
        arr, XC, YC, names=SLICE_REGIONS)
    del arr

for ch in region_arrays:
    x, y, sub = region_arrays[ch]["gulf_stream"]
    print(f"{ch:24s} gulf_stream {sub.shape}  "
          f"min {np.nanmin(sub):.3g}  max {np.nanmax(sub):.3g}")

## Section 5 — Per-field validation

- **Figure 1 — maps**: columns = the field's full dependency chain
  (raw → components → final; every computed step is shown), rows =
  validation domains.  One shared colour scale per column; land/halo
  NaNs gray; regional boxes on the global row.
- **Figure 2 — PDFs**: same grid.  Probability density; land +
  halo-rim NaNs removed; bins shared per field across domains;
  Eq. Pacific row |lat|>2° filtered for f-normalised fields.
- **Literature comparisons** live in Section 6 at the end of the
  notebook — one subsection PER REFERENCE (a reference may validate
  several fields at once), only where a reference exists.  Images in
  `../literature_figures/`, named
  `{field(s)}_{Citation}_{description}.png`.

In [ ]:
# Section 5 helpers: one call per figure, shared by all fields.
from pathlib import Path

import cartopy.crs as ccrs

from dbof.plotting.global_maps import plot_global_field
from dbof.plotting.pipeline_grids import (
    pipeline_map_grid, mask_wrap_cells, LAND_COLOR,
)
from dbof.plotting.pdfs import pipeline_pdf_grid
from dbof.plotting.literature_comparison import side_by_side

# Flat literature directory; files named
# {field}_{Citation}_{description}.png
LIT_DIR = Path("../literature_figures")

# Full dependency chain per field (columns of Figures 1-2), including
# component-level intermediates (gradient / Jacobian components).
CHAINS = {
    "relative_vorticity": ["U", "V", "dv_dx", "du_dy", "relative_vorticity"],
    "strain_n": ["U", "V", "du_dx", "dv_dy", "strain_n"],
    "strain_s": ["U", "V", "dv_dx", "du_dy", "strain_s"],
    "strain_mag": ["strain_n", "strain_s", "strain_mag"],
    "divergence": ["U", "V", "du_dx", "dv_dy", "divergence"],
    "coriolis_f": ["coriolis_f"],
    "rossby_number": ["relative_vorticity", "coriolis_f", "rossby_number"],
    "okubo_weiss": ["strain_mag", "relative_vorticity", "okubo_weiss"],
}

# f-normalised fields: |lat|>2 deg filter on the Eq. Pacific row of
# the PDFs only (maps annotated instead) — plan Clarification 8.
F_NORM = {"rossby_number"}

# Fields drawn/binned on log scales (∝-squared fields).
LOG_FIELDS = set()

PDF_NOTE = ("PDFs: density; land+rim NaNs removed; shared bins "
            "across domains; log10-x for \u221d-squared fields")


def _pdf_arrays(field):
    """Region arrays for the PDF grid of one field.

    Applies the |lat|>2 deg filter to the Eq. Pacific row for
    f-normalised chain members (filter stated in the figure title).
    Inputs: field (str).  Outputs: dict like region_arrays.
    Generated by LH and Claude
    """
    out = {}
    for f in CHAINS[field]:
        d = dict(region_arrays[f])
        if f in F_NORM:
            x, y, a = d["eq_pacific"]
            d["eq_pacific"] = (x, y,
                               np.where(np.abs(y) > 2.0, a, np.nan))
        out[f] = d
    return out


def figure1_maps(field):
    """Figure 1: dependency-chain map grid for one field.

    Inputs: field (str) — output channel name (key into CHAINS).
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    note = (" | f-normalised: Eq. Pacific extreme near equator "
            "(expected)" if field in F_NORM else "")
    pipeline_map_grid(
        CHAINS[field], region_arrays, CMAP_CFG,
        diverging_cmaps=DIVERGING, log_scale_channels=LOG_FIELDS,
        suptitle=f"Figure 1 \u2014 {field}: pipeline maps{note}",
    )
    plt.show()


def figure2_pdfs(field):
    """Figure 2: dependency-chain PDF grid for one field.

    Inputs: field (str) — output channel name (key into CHAINS).
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    note = (" | Eq. Pacific: |lat|>2\u00b0 filter (f-normalised)"
            if set(CHAINS[field]) & F_NORM else "")
    pipeline_pdf_grid(
        CHAINS[field], _pdf_arrays(field), CMAP_CFG,
        log10_fields=LOG_FIELDS,
        suptitle=f"Figure 2 \u2014 {field}: {PDF_NOTE}{note}",
    )
    plt.show()


def figure3_literature(field, png_name=None, caption=None,
                       global_view=False):
    """Figure 3: our data vs literature .png for one field.

    Inputs: field (str); png_name (str or None) — file in LIT_DIR;
    caption (str or None) — discussion text; global_view (bool) —
    render OUR panel as a global Robinson map (use when the
    literature figure is a global view) instead of the default
    Gulf Stream regional map.
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    region = "global" if global_view else "gulf_stream"

    def _render(ax):
        x, y, arr = region_arrays[field][region]
        if global_view:
            # Same seam/Arctic handling as the Figure 1 global row.
            arr = mask_wrap_cells(x, y, arr)
        ax.set_facecolor(LAND_COLOR)
        im, label = plot_global_field(
            ax, x, y, arr, field, CMAP_CFG,
            log_scale_channels=LOG_FIELDS, diverging_cmaps=DIVERGING,
            transform=ccrs.PlateCarree() if global_view else None,
            add_coastline=global_view,
            coastline_kw={"linewidth": 0.4, "edgecolor": "k"},
        )
        if im is not None:
            plt.colorbar(im, ax=ax, orientation="horizontal",
                         fraction=0.04, pad=0.04, label=label)

    side_by_side(
        _render, LIT_DIR / png_name if png_name else None,
        projection=ccrs.Robinson() if global_view else None,
        caption=caption or ("Discussion: awaiting literature "
                            f"reference for {field}."),
    )
    plt.show()

### 5.1 relative_vorticity (ζ)

ζ = dv_dx − du_dy.  Submesoscale cyclonic filaments (positive-skewed NH), eddy cores; halo rim NaN.

In [ ]:
figure1_maps("relative_vorticity")

In [ ]:
figure2_pdfs("relative_vorticity")

### 5.2 strain_n (σₙ)

σₙ = du_dx − dv_dy: frontal confluence zones.

In [ ]:
figure1_maps("strain_n")

In [ ]:
figure2_pdfs("strain_n")

### 5.3 strain_s (σₛ)

σₛ = dv_dx + du_dy: jet flanks.

In [ ]:
figure1_maps("strain_s")

In [ ]:
figure2_pdfs("strain_s")

### 5.4 strain_mag (σ)

σ = √(σₙ²+σₛ²) — positive-definite; fronts and eddy peripheries.

In [ ]:
figure1_maps("strain_mag")

In [ ]:
figure2_pdfs("strain_mag")

### 5.5 divergence (δ)

δ = du_dx + dv_dy: frontal convergence filaments; magnitudes ≪ ζ at mesoscale.

In [ ]:
figure1_maps("divergence")

In [ ]:
figure2_pdfs("divergence")

### 5.6 coriolis_f

f = 2Ω sin(lat) — analytic; validates the YC → f path.

In [ ]:
figure1_maps("coriolis_f")

In [ ]:
figure2_pdfs("coriolis_f")

### 5.7 rossby_number (Ro = ζ/f)

O(0.1–1) in energetic regions; equatorial blow-up expected (f→0; PDF row filtered).

In [ ]:
figure1_maps("rossby_number")

In [ ]:
figure2_pdfs("rossby_number")

### 5.8 okubo_weiss (W)

W = σ² − ζ²: negative eddy cores, positive strain peripheries.

In [ ]:
figure1_maps("okubo_weiss")

In [ ]:
figure2_pdfs("okubo_weiss")

## Section 6 — Literature comparisons

One subsection per reference (a reference may validate several fields); only fields with published counterparts appear here.

### 6.1 Bachman et al. (2021) — ζ (ω) and W over the Kerguelen Plateau

Reference figure shows b, ω, W, W* (60–85°E, 50–38°S).  This subset
can reproduce the ζ (their ω) and W (okubo_weiss) panels; b is
validated in `frontal_structure.ipynb` and W* in
`frontogenesis.ipynb` (cross-reference rule).  Different model/date —
compare structure and magnitudes, not features.

In [ ]:
# Bachman et al. (2021) comparison: stacked maps of zeta and
# okubo_weiss over the Kerguelen Plateau.
from dbof.plotting.literature_comparison import stacked_side_by_side

KREG = "kerguelen"


def _kerg(field):
    """Kerguelen slice of one field.  Generated by LH and Claude."""
    return region_arrays[field][KREG]


def _kerg_panel(field):
    """Renderer for one Kerguelen map panel.

    Inputs: field (str) — field_cmaps key present in region_arrays.
    Outputs: callable(ax).  Generated by LH and Claude
    """
    x, y, arr = _kerg(field)

    def _render(ax):
        ax.set_facecolor(LAND_COLOR)
        im, label = plot_global_field(
            ax, x, y, arr, field, CMAP_CFG,
            diverging_cmaps=DIVERGING, add_coastline=False)
        if im is not None:
            plt.colorbar(im, ax=ax, label=label, fraction=0.04)
        ax.set_xticks([])
        ax.set_yticks([])
    return _render


stacked_side_by_side(
    [_kerg_panel("relative_vorticity"),
     _kerg_panel("okubo_weiss")],
    LIT_DIR / ("vorticity-okuboweiss_Bachman-etal(2021)_"
               "Kerguelen-maps.png"),
    our_titles=["relative_vorticity (ω)", "okubo_weiss (W)"],
    caption=("Bachman et al. (2021) panels b (ω) and c (W).  "
             "Expect eddy cores (W<0) ringed by strain (W>0); "
             "b → frontal_structure.ipynb, "
             "W* → frontogenesis.ipynb."),
)
plt.show()

### 6.2 Balwada et al. (2021) — vorticity–strain joint PDFs

Reference setup: an IDEALISED ACC-like channel (2000 km × 2000 km ×
3 km, β-plane centred at 35°S, meridional Gaussian ridge 1 km high /
150 km wide), with f₀ = f(35°S) < 0.  Our Kerguelen box is a
reasonable real-ocean analog: comparable size (~1900 × 1330 km) and
a real topographic ridge in the ACC.

SIGN CONVENTION (matters in the SH): ζ is normalised by SIGNED f₀
(regional mean f, negative here), σ and Δ by |f₀| — so CYCLONIC
vorticity (ζ < 0 in the SH) maps to POSITIVE ζ/f₀, as in the
reference.  Normalising by |f| instead mirrors the plots left-right.

Panels: occurrence JPDF, conditional-mean divergence Δ̄/|f₀|, and
conditional-mean |∇b| (computed live from Theta/Salt via
``compute_buoyancy_gradients``).  ``dbof.plotting.jpdfs`` is a fresh
implementation — reconcile against
``fronts/properties/analysis/jpdf.py``.

In [ ]:
# Balwada et al. (2021)-style joint PDFs over the Kerguelen box.
from dbof.plotting.jpdfs import (
    plot_jpdf_occurrence, plot_jpdf_conditional,
    plot_jpdf_conditional_log,
)

_zeta = _kerg("relative_vorticity")[2]
_sig = _kerg("strain_mag")[2]
_div = _kerg("divergence")[2]
_gb = _kerg("gradb_mag")[2]

# SIGNED f0 (Balwada: f0 = f(35S) < 0).  Cyclones (zeta<0 in the
# SH) then map to POSITIVE zeta/f0; using |f| mirrors the plots.
f0 = np.nanmean(_kerg("coriolis_f")[2])
print(f"f0 (signed regional mean f) = {f0:.3e} s-1")

_m = (np.isfinite(_zeta) & np.isfinite(_sig)
      & np.isfinite(_div) & np.isfinite(_gb))
zf = _zeta[_m] / f0
sf = _sig[_m] / np.abs(f0)
dv = _div[_m] / np.abs(f0)
gb = _gb[_m]


def _occ(ax):
    plot_jpdf_occurrence(ax, zf, sf)


side_by_side(
    _occ,
    LIT_DIR / ("jpdf-occurrence_Balwada-etal(2021)_"
               "vorticity-strain.png"),
    caption=("Occurrence JPDF, signed-f0 convention: expect the "
             "Balwada et al. (2021) shape — mode near the "
             "origin, cyclonic (right) tail along the σ = "
             "ζ separatrix.  Reference is an idealised ACC "
             "channel; compare shape and regime occupation."),
)
plt.show()


def _cond(ax):
    plot_jpdf_conditional(ax, zf, sf, dv,
                          clabel=r"$\overline{\Delta}/|f_0|$")


side_by_side(
    _cond,
    LIT_DIR / ("jpdf-divergence_Balwada-etal(2021)_"
               "vorticity-strain.png"),
    caption=("Conditional-mean divergence: expect convergence "
             "(negative, blue) concentrated along the cyclonic "
             "strain ridge — Balwada et al. (2021)."),
)
plt.show()


def _condgb(ax):
    plot_jpdf_conditional_log(
        ax, zf, sf, gb,
        clabel=r"$\overline{|\nabla b|}$ (s$^{-2}$)")


side_by_side(
    _condgb,
    LIT_DIR / ("jpdf-gradb_Balwada-etal(2021)_"
               "vorticity-strain.png"),
    caption=("Conditional-mean |∇b| (log scale, 1e-8–"
             "1e-6): expect the tightest buoyancy gradients along "
             "the cyclonic-strain ridge, weakest in the AVD "
             "interior — Balwada et al. (2021)."),
)
plt.show()

## Summary — guard-style checks

Pass/fail checklist mirroring `dev/verify_subsets_real_data.py`:
finite fraction, plausible range, and channel-uniqueness guard on the
Gulf Stream domain.

In [ ]:
# Summary: quick stats + uniqueness guard (Gulf Stream domain).
seen = {}
print(f"{'field':24s} {'finite%':>8s} {'min':>11s} "
      f"{'max':>11s} {'unique':>7s}")
for ch in CHANNELS:
    sub = region_arrays[ch]["gulf_stream"][2]
    finite = np.isfinite(sub)
    frac = 100.0 * finite.mean()
    key = hash(sub[finite][::997].tobytes()) if finite.any() else ch
    dup = seen.get(key)
    seen[key] = ch
    print(f"{ch:24s} {frac:7.1f}% {np.nanmin(sub):11.3g} "
          f"{np.nanmax(sub):11.3g} {'DUP!' if dup else 'ok':>7s}")
    assert dup is None, f"{ch} identical to {dup}"
print("\nAll checks passed.")

In [ ]:
# Store-vs-live consistency: each final channel (store) must equal
# the function of its live-computed dependencies (Gulf Stream domain;
# land + halo-rim NaNs excluded).  This turns the two-path design
# (finals via RUN->store->LOAD, dependencies via live compute) into
# an explicit pass/fail test of the pipeline plumbing.
from dbof.preprocessing.physical_constants import (
    G, RHO0_REFERENCE, ALPHA, BETA,
)


def _gs(field):
    """Gulf Stream slice of one field.

    Inputs: field (str).  Outputs: 2D np.ndarray.
    Generated by LH and Claude
    """
    return region_arrays[field]["gulf_stream"][2]


CHECKS = {
    "relative_vorticity = dv_dx - du_dy":
        (_gs("relative_vorticity"), _gs("dv_dx") - _gs("du_dy")),
    "strain_n = du_dx - dv_dy":
        (_gs("strain_n"), _gs("du_dx") - _gs("dv_dy")),
    "strain_s = dv_dx + du_dy":
        (_gs("strain_s"), _gs("dv_dx") + _gs("du_dy")),
    "divergence = du_dx + dv_dy":
        (_gs("divergence"), _gs("du_dx") + _gs("dv_dy")),
    "strain_mag = sqrt(strain_n^2 + strain_s^2)":
        (_gs("strain_mag"),
         np.sqrt(_gs("strain_n")**2 + _gs("strain_s")**2)),
    "okubo_weiss = strain_mag^2 - vorticity^2":
        (_gs("okubo_weiss"),
         _gs("strain_mag")**2 - _gs("relative_vorticity")**2),
    "rossby_number = vorticity / f":
        (_gs("rossby_number"),
         _gs("relative_vorticity") / _gs("coriolis_f")),
}

REL_TOL = 1e-4       # float32 store vs float64 live recompute
for name, (store_v, recomputed) in CHECKS.items():
    m = np.isfinite(store_v) & np.isfinite(recomputed)
    scale = max(float(np.nanmax(np.abs(recomputed[m]))), 1e-300)
    rel = float(np.nanmax(np.abs(store_v[m] - recomputed[m]))) / scale
    flag = "OK  " if rel < REL_TOL else "FAIL"
    print(f"{flag} {name}  (max rel err {rel:.2e}, n={m.sum()})")
    assert rel < REL_TOL, name
print("\nStore-vs-live consistency: all checks passed.")

**Cross-references** — rotation/Jacobian machinery is validated HERE (sibling notebooks reference this one); rotated U/V as output channels → `native_fields.ipynb`; W* (∇b-modified Okubo-Weiss) → `frontogenesis.ipynb`.